# <p style = "text-align: center">Data to equations</p>

## <p style = "text-align: center">An introduction to sparse identification of nonlinear dynamics</p>

Sometimes all we have at our disposal to study a system is data. This can be because the system under consideration is too complex to be modeled starting from first principles, or because we don't know what phenomena are at play. Other times we do have some idea of what our system equations should look like, but don't know the exact dynamics. This can for exmaple be a nonlinear damping or spring that turns out to be difficult to model. In both cases, we may be interested in deriving a set of governing equations directly from the data, in the hopes of gaining a more fundamental understanding of the system under consideration.

## 0) Set up work space

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
import pysindy as ps

import simulation_tools as simt
from importlib import reload

In [ ]:
reload(simt)

## 1) Background and mathematics

In this introductory notebook, we will try to find the governing equations for time-invariant systems with smooth dynamics, where we do not yet consider the control input. These systems are governed by the following equation:

$$
\dot{x} = f(x) \quad \textrm{with} \quad x \in \mathbb{R}^n
$$

Throughout this section, the state $x$ and its derivative are expressed as a **row** vectors. Finding the exact system from data is extremely challenging as there are uncountably infinitely many possible forms for the dynamics of each state, and the problem becomes combinatorially harder for increasing dimension of the state space $n$. However, things become ridiculously easy if we restrict ourselves to approximating the dynamics $f$ by a generalized linear model. A generalized linear model is a linear regression on (nonlinear) functions of the variables under consideration. We can express this using the following equations:

$$
\begin{aligned}
    &\text{for one state variable:} \qquad & \dot{x}_i = f_i(x) &\approx \sum_{k=1}^{p} \Theta_k(x) \Xi_{ki} \\
    &\text{for entire state:} \qquad & \dot{x} = f(x) &\approx \Theta(x) \Xi
\end{aligned}
$$

So we express $f$ as a linear combination of $p$ possible nonlinear candidate functions of the state, given by the row vector $\Theta(x)$. The coefficients are given by $\Xi$. We will then try to find the coefficients using sparse regression on the available data. Sparse regression means that we look for as little non-zero coefficients as possible, by using an $l_1$-regularization.

Now, we will first discuss how we order the available data for the regression. We assume that we have a sequence of measurements of the state and of the derivative of the state, both of length $m$. In practice, the derivatives will be calculated numerically from the measurements of the state, so you should be careful not to introduce too much noise and use a numerically robust differentiation scheme. We will then organize them into two data matrices $X$ and $\dot{X} \in \mathbb{R}^{m\times n}$, as follows:

$$
X = 
\begin{bmatrix}
    x(t_1) \\
    x(t_2) \\
    \vdots \\
    x(t_m)
\end{bmatrix}, \quad
\dot{X} = 
\begin{bmatrix}
    \dot{x}(t_1) \\
    \dot{x}(t_2) \\
    \vdots \\
    \dot{x}(t_m)
\end{bmatrix}
$$

We then also calculate the candidate functions evaluated in the measured states, this can for example look like:

$$
\Theta(X) = 
\begin{bmatrix}
    1 & X & X^2 & \dots & X^d & \dots & sin(X) & cos(X) & \dots
\end{bmatrix}
$$

$\Theta(X) \in \mathbb{R}^{m\times p}$ can contain any nonlinear functions of the data matrix $X$ (element-wise evaluation) you can imagine. Usually polynomials up to a certain degree are at least part of the library of cadidate functions. $X^d$ denotes the matrix with column vectors given by all possible polynomials of degree $d$ in the state $x$, so: $\prod_{i} x_i^{q_i}$ with $\sum_{i} q_i = d$. Given these all these data matrices, we will then try to determine $\Xi$ such that the following expression approximately holds:

$$
\dot{X} = \Theta(X)\Xi
$$

We do this by solving the following optimization problem, where $\xi_k$ are the columns of $\Xi$:

$$
\xi_k^* = \text{arg}\,\min\limits_{\xi_k}\, || \dot{X}_k - \Theta(X)\xi_k||_2 + \lambda||\xi_k||_1
$$

Increasing the value of $\lambda$ will lead to a sparser model for the dynamics, decreasing it will lead to lower errors. A proper value should be chosen so that enough of the variation in the data is explained, yet there is no overfitting (to the noise for example). This value can be chosen using information criteria. After this optimization, our identified dynamical system is given by:

$$
\dot{x} = \Theta(x)\Xi^*
$$

The core concept of this enire apporach is simple and by no means new, but due to recent advances in numerically robust algorithms for differentiation and sparse regression, this simple approach can work wonderfully on real-world data.

References:
- S. L. Brunton and J. N. Kutz, Data-Driven Science and Engineering: Machine Learning, Dynamical Systems, and Control, 2nd ed. Cambridge: Cambridge University Press, 2022. 

## 2) Applications with pySINDy

### A) The Lorenz system

In [ ]:
# A noisy Lorenz system
v = simt.StationaryGaussianWhiteNoise(n=3, mu=np.zeros(3), cov=np.diag(np.ones(3)*100), seed=83)
LorenzSystem = simt.ContinuousNonlinearTimeInvariant(f=simt.lorenz, n=3, v=v, x0=np.ones(3))

# Generate data
fs = 100
dt = 1/fs
npts = 100000
T = (npts-1)*dt
t, x = LorenzSystem.run(T, num=npts)

In [ ]:
# Plot: see that it is very noisy indeed
start_plot = 1000
len_plot = 1000
plot_idxs = np.arange(start_plot, start_plot+len_plot, dtype=int)

fig, ax = plt.subplots(subplot_kw={'projection': '3d'})

ax.scatter(x[0, plot_idxs], x[1, plot_idxs], x[2, plot_idxs], c=t[plot_idxs], cmap="plasma")
ax.set_proj_type('ortho')
ax.set_xticks([-20, -10, 0, 10, 20])
ax.set_yticks([-20, -10, 0, 10, 20])
ax.set_zticks([10, 20, 30, 40])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

plt.show()

In [ ]:
# Initiate model
differentiation_method = ps.FiniteDifference(order=2)
feature_library = ps.PolynomialLibrary(degree=3)
optimizer = ps.STLSQ(threshold=0.1)

nonlinear_model = ps.SINDy(
    differentiation_method=differentiation_method,
    feature_library=feature_library,
    optimizer=optimizer,
)

# Fit model
nonlinear_model.fit(np.transpose(x), dt, feature_names=["x", "y", "z"])

# Show the identified model
nonlinear_model.print()

Very close to the underlying system:

$$
\begin{aligned}
    \dot{x} &= - 10x + 10y \\
    \dot{y} &= 28x - y - xz \\
    \dot{z} &= -\frac{8}{3}z + xy
\end{aligned}
$$

What if we don't have as much data?

In [ ]:
# Initiate model
differentiation_method = ps.FiniteDifference(order=2)
feature_library = ps.PolynomialLibrary(degree=3)
optimizer = ps.STLSQ(threshold=0.1)

nonlinear_model = ps.SINDy(
    differentiation_method=differentiation_method,
    feature_library=feature_library,
    optimizer=optimizer,
)

# Fit model
# nonlinear_model.fit(np.transpose(x)[:10000], dt, feature_names=["x", "y", "z"]) # Redo with 10th of the data
nonlinear_model.fit(np.transpose(x)[:1000], dt, feature_names=["x", "y", "z"]) # Redo with 100th of the data

# Show the identified model
nonlinear_model.print()

Performance deteriorates for decreasing amount of training data, and also for decreasing training data quality.

### B) The Duffing oscillator

In [ ]:
# Duffing equations
def duffing(x):
    return np.array([x[1], x[0]-x[0]**3])

# A noisy Duffing oscillator system
v = simt.StationaryGaussianWhiteNoise(n=2, mu=np.zeros(2), cov=np.diag(np.ones(2)*10), seed=83)
DuffingSystem = simt.ContinuousNonlinearTimeInvariant(f=duffing, n=2, v=v, x0=np.ones(2))

# Generate data
fs = 100
dt = 1/fs
npts = 100000
T = (npts-1)*dt
t, x = DuffingSystem.run(T, num=npts)

In [ ]:
# Plot
start_plot = 0
len_plot = 1000
plot_idxs = np.arange(start_plot, start_plot+len_plot, dtype=int)

fig, ax = plt.subplots()

ax.scatter(x[0, plot_idxs], x[1, plot_idxs], c=t[plot_idxs])

fig.tight_layout()
plt.show()

In [ ]:
# Initiate model
differentiation_method = ps.FiniteDifference(order=2)
feature_library = ps.PolynomialLibrary(degree=3)
optimizer = ps.STLSQ(threshold=0.1)

nonlinear_model = ps.SINDy(
    differentiation_method=differentiation_method,
    feature_library=feature_library,
    optimizer=optimizer,
)

# Fit model
nonlinear_model.fit(np.transpose(x), dt, feature_names=["x", "y", "z"])
# nonlinear_model.fit(np.transpose(x)[:1000], dt, feature_names=["x", "y", "z"])

# Show the identified model
nonlinear_model.print()

Very close to the underlying system:

$$
\begin{aligned}
    \dot{x} &= y \\
    \dot{y} &= x - x^3
\end{aligned}
$$

### C) Where it fails: identifying a nonlinear system with insufficient data

In [ ]:
# Load data
data_arr = np.loadtxt("data/Impact_SDOF_HardeningNES.csv", delimiter=",", skiprows=1) # time, impact force, acceleration absorber mass, acceleration, main system mass

# Plot data
fig, axs = plt.subplots(nrows=3, figsize=(10, 10))

axs[0].plot(data_arr[:, 0], data_arr[:, 1])
axs[0].set_title("Impact force")
axs[0].set_xlabel("Time [s]")
axs[0].set_ylabel("Force [N]")

axs[1].plot(data_arr[:, 0], data_arr[:, 3])
axs[1].set_title("Main system mass acceleration")
axs[1].set_xlabel("Time [s]")
axs[1].set_ylabel("Acceleration [m/s$^2$]")

axs[2].plot(data_arr[:, 0], data_arr[:, 2])
axs[2].set_title("Absorber mass acceleration")
axs[2].set_xlabel("Time [s]")
axs[2].set_ylabel("Acceleration [m/s$^2$]")

fig.tight_layout()
plt.show()

In [ ]:
###############################################################################
# Select regions of data suitable for training: no impact, vibrations present #
###############################################################################

# Reorganize data
time_arr = data_arr[:, 0].flatten()
force_arr = data_arr[:, 1].flatten()
absorber_arr = data_arr[:, 2].flatten()
system_arr = data_arr[:, 3].flatten()

npts = time_arr.size
dt = (time_arr[-1]-time_arr[0])/(npts-1)
fs = 1/dt
print(f"The sampling frequency is {int(fs)} Hz.")

# Find masks
sos = butter(2, 1, btype="low", analog=False, output="sos", fs=fs) # Low-pass filter
absorber_envelope_arr = sosfiltfilt(sos, np.abs(absorber_arr)) # Apply low-pass filter to absorber acceleration
sign_vibr_mask = (absorber_envelope_arr > 1)

sign_force_mask = (force_arr > 100) # Threshold

# Widen force peaks
force_peak_edges = np.nonzero(np.diff(sign_force_mask))[0]

wide_edges_start = force_peak_edges[[i%2 == 0 for i in np.arange(force_peak_edges.size)]]-2000 # to compensate for slow rising of edges
wide_edges_end = force_peak_edges[[i%2 != 0 for i in np.arange(force_peak_edges.size)]]+500 # for safety

sign_force_mask_wide = np.zeros_like(sign_force_mask)
for i in np.arange(wide_edges_start.size):
    sign_force_mask_wide[wide_edges_start[i]:wide_edges_end[i]] = 1

# Combine both masks
good_data_mask = np.logical_and(sign_vibr_mask, np.logical_not(sign_force_mask_wide))
print(f"Only {np.sum(good_data_mask)/fs:.4g} seconds of usable data available.")

# Plot acceleration, envelope and masks
fig, ax = plt.subplots(nrows=1, figsize=(10, 5))

ax.plot(time_arr, absorber_arr, label="Raw acceleration")
ax.plot(time_arr, absorber_envelope_arr, label="Envelope")
ax.plot(time_arr, sign_vibr_mask*50, label="Significant vibrations")
ax.plot(time_arr, sign_force_mask_wide*50, label="Significant forces")
ax.plot(time_arr, good_data_mask*100, label="Valid training data")
ax.set_title("Absorber acceleration, envelope and masks")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Acceleration [m/s$^2$]")
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
###############################################
# Calculate state variables from measurements #
###############################################

sos_integration = butter(2, [1, 25], btype="bandpass", analog=False, output="sos", fs=fs)

absorber_acc_filt = sosfiltfilt(sos_integration, absorber_arr)
system_acc_filt = sosfiltfilt(sos_integration, system_arr)
absorber_vel = np.cumsum(absorber_acc_filt)*dt
system_vel = np.cumsum(system_acc_filt)*dt

absorber_vel_filt = sosfiltfilt(sos_integration, absorber_vel)
system_vel_filt = sosfiltfilt(sos_integration, system_vel)
absorber_pos = np.cumsum(absorber_vel_filt)*dt
system_pos = np.cumsum(system_vel_filt)*dt

absorber_pos_filt = sosfiltfilt(sos_integration, absorber_pos)
system_pos_filt = sosfiltfilt(sos_integration, system_pos)

# Plot
fig, axs = plt.subplots(nrows=4, figsize=(10, 13))

plot_start = 25*fs
plot_time = 5*fs
plot_idxs = np.arange(plot_start, plot_start+plot_time, dtype=int)
plot_start = 0
plot_time = len(absorber_arr)
plot_idxs = np.arange(plot_start, plot_start+plot_time, dtype=int)

axs[0].plot(time_arr[plot_idxs], absorber_pos_filt[plot_idxs])
axs[0].set_title("Absorber position")
axs[0].set_xlabel("Time [s]")
axs[0].set_ylabel("Position [m]")

axs[1].plot(time_arr[plot_idxs], system_pos_filt[plot_idxs])
axs[1].set_title("System position")
axs[1].set_xlabel("Time [s]")
axs[1].set_ylabel("Position [m]")

axs[2].plot(time_arr[plot_idxs], absorber_vel_filt[plot_idxs])
axs[2].set_title("Absorber velocity")
axs[2].set_xlabel("Time [s]")
axs[2].set_ylabel("Velocity [m/s]")

axs[3].plot(time_arr[plot_idxs], system_vel_filt[plot_idxs])
axs[3].set_title("System velocity")
axs[3].set_xlabel("Time [s]")
axs[3].set_ylabel("Velocity [m/s]")

fig.tight_layout()
plt.show()


In [ ]:
#################################
# Identify system using pysindy #
#################################

# Restructure data once more
x = np.concat((absorber_pos_filt.reshape((-1, 1)), 
               absorber_vel_filt.reshape((-1, 1)), 
               system_pos_filt.reshape((-1, 1)), 
               system_vel_filt.reshape((-1, 1))), axis=1)

# Normalize
x_norm_factor = 1
x_norm = x/x_norm_factor

# Only retain good chunks
x_train = []
for i in np.arange(wide_edges_start.size):
    x_train.append(x_norm[wide_edges_start[i]:wide_edges_end[i], :])

dt_train = [dt for i in np.arange(len(x_train))]

# Initiate model
differentiation_method = ps.FiniteDifference(order=4)
feature_library = ps.PolynomialLibrary(degree=3, include_bias=False, include_interaction=True)
# def f1_name(x):
#     return x
# def f2_name(x):
#     return x + "^3"
# def f3_name(x, y):
#     return y + "*" + x + "^2"
# feature_library = ps.CustomLibrary([lambda x: x, lambda x: x**3, lambda x, y: y*x**2], function_names=[f1_name, f2_name, f3_name])
optimizer = ps.STLSQ(threshold=0.5)

nonlinear_model = ps.SINDy(
    differentiation_method=differentiation_method,
    feature_library=feature_library,
    optimizer=optimizer,
)

# Fit model
tic = time.perf_counter()
nonlinear_model.fit(x_train, dt_train, feature_names=["z", "z'", "x", "x'"])
toc = time.perf_counter()

# Show the identified model
nonlinear_model.print()

The identified model makes no sense at all.